# 06 - Entendiendo el DeepPiCar: El Proyecto que Inspiró MLForge

Este notebook analiza el proyecto original de coche autónomo con Raspberry Pi + Google Coral.
Entender cómo funciona nos da perspectiva sobre por qué creamos MLForge.

## Contenido
1. Arquitectura del sistema
2. Lane following: behavior cloning con CNN
3. Object detection: SSD MobileNet en Edge TPU
4. Pipeline completo: cámara → decisión → control
5. Lecciones aprendidas
6. Cómo MLForge mejora este flujo

## 1. Arquitectura del Sistema

El DeepPiCar es un coche a escala controlado por IA que combina dos sistemas:

```
┌──────────────────────────────────────────────────────┐
│                    Raspberry Pi 4                     │
│                                                       │
│  ┌─────────────┐    ┌──────────────┐                 │
│  │   Cámara     │───→│  OpenCV       │                │
│  │   USB/CSI    │    │  (captura)    │                │
│  └─────────────┘    └──────┬───────┘                 │
│                             │                         │
│                    ┌────────┴────────┐                │
│                    ▼                 ▼                │
│  ┌──────────────────┐  ┌──────────────────┐         │
│  │ Lane Following    │  │ Object Detection  │         │
│  │ (Keras CNN)       │  │ (SSD MobileNet)   │         │
│  │ CPU inference     │  │ Coral Edge TPU    │         │
│  └────────┬─────────┘  └────────┬─────────┘         │
│           │ ángulo              │ objetos             │
│           └────────┬────────────┘                     │
│                    ▼                                  │
│           ┌────────────────┐                         │
│           │ Decision Logic  │                         │
│           │ (Python)        │                         │
│           └────────┬───────┘                         │
│                    ▼                                  │
│           ┌────────────────┐                         │
│           │ PiCar Motors    │                         │
│           │ (steering+speed)│                         │
│           └────────────────┘                         │
└──────────────────────────────────────────────────────┘
```

### Hardware:
- **Raspberry Pi 3B+/4**: cerebro central
- **Google Coral USB Accelerator**: acelera detección de objetos
- **SunFounder PiCar-S**: kit de coche con servo (dirección) y motor DC
- **Cámara USB/CSI**: captura road images a 320×240

## 2. Lane Following: Behavior Cloning

El coche aprende a seguir carriles imitando a un conductor humano.
Esta técnica se llama **Behavior Cloning** (aprendizaje por imitación).

### Recolección de datos:
1. Un humano conduce el coche por una pista
2. El sistema guarda: imagen de la cámara + ángulo del volante
3. Se generan ~5000-10000 pares (imagen, ángulo)

### Modelo: CNN End-to-End
```
Imagen (320×240×3) → CNN → ángulo de volante (-1.0 a 1.0)
```

Inspirado en el paper de NVIDIA "End to End Learning for Self-Driving Cars" (2016).

In [ ]:
# Veamos el modelo de lane following original
import os
import sys

# Ruta al proyecto original
DEEPPICAR_DIR = os.path.abspath('../..')
MODEL_PATH = os.path.join(
    DEEPPICAR_DIR, 
    'DeepPiCar-master', 'models', 'lane_navigation', 'data', 
    'model_result', 'lane_navigation.h5'
)

print(f"Buscando modelo en: {MODEL_PATH}")
print(f"Existe: {os.path.exists(MODEL_PATH)}")

if os.path.exists(MODEL_PATH):
    size_mb = os.path.getsize(MODEL_PATH) / 1024 / 1024
    print(f"Tamaño: {size_mb:.2f} MB")

In [ ]:
# Intentar cargar y analizar el modelo
try:
    import tensorflow as tf
    
    if os.path.exists(MODEL_PATH):
        model = tf.keras.models.load_model(MODEL_PATH)
        model.summary()
        
        total_params = model.count_params()
        print(f"\nTotal parámetros: {total_params:,}")
        print(f"\nEste modelo toma una imagen y predice un ángulo de volante.")
        print("Es un problema de REGRESIÓN (no clasificación).")
    else:
        print("Modelo no encontrado. Esto es normal si el proyecto original")
        print("no está en la misma carpeta.")
        print("\nLa arquitectura típica de NVIDIA PilotNet es:")
        print("  Conv2D(24, 5x5) → Conv2D(36, 5x5) → Conv2D(48, 5x5)")
        print("  → Conv2D(64, 3x3) → Conv2D(64, 3x3)")
        print("  → Flatten → Dense(100) → Dense(50) → Dense(10) → Dense(1)")
        print("  Output: ángulo de volante (float)")

except ImportError:
    print("TensorFlow no disponible. La arquitectura del modelo es:")
    print("NVIDIA PilotNet - 5 capas convolucionales + 4 capas densas")
    print("Input: 200×66×3 (imagen recortada de la carretera)")
    print("Output: 1 valor float (ángulo del volante)")

### El Preprocesamiento de la Imagen

```python
# Del código original (hand_coded_lane_follower.py):
def detect_lane(frame):
    # 1. Convertir a escala de grises
    gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
    
    # 2. Gaussian blur para reducir ruido
    blur = cv2.GaussianBlur(gray, (5, 5), 0)
    
    # 3. Canny edge detection
    edges = cv2.Canny(blur, 50, 150)
    
    # 4. Region of Interest (solo la carretera)
    mask = create_roi_mask(edges)
    masked = cv2.bitwise_and(edges, mask)
    
    # 5. Hough Lines para detectar líneas de carril
    lines = cv2.HoughLinesP(masked, ...)
    
    # 6. Calcular ángulo de steering
    angle = compute_steering_angle(lines)
    return angle
```

Este es el approach **hand-coded** (reglas manuales). El approach **end-to-end** (CNN) aprende todo esto directamente de los datos, sin programar reglas.

## 3. Object Detection: SSD MobileNet en Edge TPU

El sistema de detección identifica señales de tráfico y peatones usando
un modelo SSD MobileNet V2 ejecutado en el **Google Coral Edge TPU**.

### Modelos usados:
| Modelo | Tarea | Formato |
|--------|-------|---------|
| `mobilenet_ssd_v2_coco_quant_postprocess_edgetpu.tflite` | Detección COCO (80 clases) | Edge TPU |
| `road_signs_quantized_edgetpu.tflite` | Señales de tráfico custom | Edge TPU |
| `road_signs_quantized.tflite` | Señales de tráfico (CPU fallback) | TFLite INT8 |

### Código de inferencia (del proyecto original):
```python
# objects_on_road_processor.py
from edgetpu.detection.engine import DetectionEngine

class ObjectsOnRoadProcessor:
    def __init__(self):
        self.engine = DetectionEngine(
            'road_signs_quantized_edgetpu.tflite'
        )
    
    def process_frame(self, frame):
        # Detectar objetos
        objects = self.engine.detect_with_image(
            frame, threshold=0.5, top_k=3
        )
        
        for obj in objects:
            label = self.labels[obj.label_id]
            score = obj.score
            bbox = obj.bounding_box
            
            # Tomar decisión según el objeto
            if label == 'stop_sign':
                self.set_speed(0)
            elif label == 'speed_limit_40':
                self.set_speed(40)
            elif label == 'person':
                self.set_speed(0)  # STOP!
```

El Edge TPU permite ejecutar la detección en **~10ms por frame** (100 FPS teóricos),
mientras que en CPU de RPi tardaría ~200-300ms.

## 4. Pipeline Completo

```python
# deep_pi_car.py — Loop principal simplificado
while running:
    # 1. Capturar frame de la cámara
    frame = camera.read()
    
    # 2. Lane following: obtener ángulo de dirección
    steering_angle = lane_follower.follow_lane(frame)
    
    # 3. Object detection: detectar señales/peatones
    objects = object_detector.process_frame(frame)
    
    # 4. Lógica de decisión
    speed = compute_speed(objects)  # Frenar si hay stop/persona
    
    # 5. Actuar: mover motores
    car.steer(steering_angle)
    car.set_speed(speed)
```

### Flujo de datos:
```
Cámara (30fps) → Frame → [Lane CNN (CPU)] → ángulo
                       → [SSD (Edge TPU)] → objetos → speed
                                                     → steering
                                                     → PiCar motors
```

In [ ]:
# Veamos los datos de entrenamiento originales
import glob

data_dir = os.path.join(DEEPPICAR_DIR, 'DeepPiCar-master', 'driver', 'data')

if os.path.exists(data_dir):
    images = glob.glob(os.path.join(data_dir, '*.png')) + \
             glob.glob(os.path.join(data_dir, '*.jpg'))
    print(f"Imágenes de entrenamiento encontradas: {len(images)}")
    
    if images:
        # Mostrar algunas
        import matplotlib.pyplot as plt
        from PIL import Image
        
        fig, axes = plt.subplots(1, min(5, len(images)), figsize=(15, 3))
        if not isinstance(axes, np.ndarray):
            axes = [axes]
        for ax, img_path in zip(axes, images[:5]):
            img = Image.open(img_path)
            ax.imshow(img)
            ax.set_title(os.path.basename(img_path)[:20], fontsize=8)
            ax.axis('off')
        plt.suptitle('Imágenes de la cámara del coche')
        plt.tight_layout()
        plt.show()
        
        # Info de una imagen
        img = Image.open(images[0])
        print(f"Resolución: {img.size}")
        print(f"Formato: {img.mode}")
else:
    print(f"Directorio de datos no encontrado: {data_dir}")
    print("Las imágenes de entrenamiento son capturas de 320x240")
    print("del punto de vista de la cámara frontal del coche.")

In [ ]:
# Analizar los modelos TFLite del Edge TPU
tflite_models_dir = os.path.join(
    DEEPPICAR_DIR, 'DeepPiCar-master', 'models', 
    'object_detection', 'data', 'model_result'
)

if os.path.exists(tflite_models_dir):
    print("Modelos TFLite del proyecto original:\n")
    for f in os.listdir(tflite_models_dir):
        if f.endswith('.tflite'):
            path = os.path.join(tflite_models_dir, f)
            size = os.path.getsize(path) / 1024 / 1024
            is_edgetpu = 'edgetpu' in f
            print(f"  {f}")
            print(f"    Tamaño: {size:.2f} MB")
            print(f"    Edge TPU: {'Sí' if is_edgetpu else 'No (CPU only)'}")
            print()
else:
    print("Modelos típicos del proyecto:")
    print("  mobilenet_ssd_v2_coco_quant_postprocess_edgetpu.tflite (~4 MB)")
    print("  road_signs_quantized_edgetpu.tflite (~2 MB)")
    print("  road_signs_quantized.tflite (~2 MB)")

## 5. Lecciones Aprendidas del DeepPiCar

### Lo que funciona bien:
- **Behavior cloning** es sorprendentemente efectivo para tareas simples de control
- **Edge TPU** da un speedup masivo (~10-20x) para inferencia INT8
- **Transfer learning** (SSD MobileNet pre-entrenado en COCO) funciona con pocos datos custom
- **Separar percepción y control** hace el sistema más mantenible

### Limitaciones:
- **Todo es manual**: no hay pipeline automatizado para entrenar/exportar
- **Un solo modelo**: cambiar de MobileNet a EfficientNet requiere reescribir código
- **No hay tracking**: no se guardan métricas de experimentos
- **Deployment hardcoded**: cada plataforma tiene su propio código ad-hoc
- **Datos frágiles**: sin augmentation, sin validación de calidad

### Conceptos que aplican a cualquier proyecto de ML:

| Concepto | DeepPiCar | MLForge |
|----------|-----------|----------|
| Recolección datos | Manual con cámara | `data.source: local/torchvision/huggingface` |
| Augmentation | Ninguna | `data.augmentation: [flip, rotate, ...]` |
| Modelo | Keras manual | `model.architecture: mobilenet_v3_small` |
| Training | Script único | `mlforge train --config config.yaml` |
| Export | Manual TFLite | `mlforge export --formats all` |
| Benchmark | No existe | `mlforge benchmark --model-dir exports/` |
| Deploy | Código ad-hoc | `mlforge deploy android/ios/web/edge` |
| Tracking | No existe | Dashboard web con métricas en tiempo real |

## 6. De DeepPiCar a MLForge

MLForge nació de la pregunta: **¿cómo sería el DeepPiCar si tuviera un framework profesional detrás?**

### Ejemplo: rehacer el clasificador de señales con MLForge

```yaml
# road_signs_config.yaml
project:
  name: road_sign_detector
  task: classification

data:
  source: local
  path: ./road_signs_dataset/
  input_size: 224
  augmentation:
    - horizontal_flip
    - random_rotation: 15
    - color_jitter: {brightness: 0.3, contrast: 0.3}
    - gaussian_blur: {kernel_size: 3}

model:
  architecture: mobilenet_v3_small  # Mismo tipo que el original
  pretrained: true
  num_classes: 5  # stop, speed_40, speed_60, person, green_light

training:
  epochs: 30
  batch_size: 32
  optimizer: adam
  learning_rate: 0.001
  scheduler: cosine
  early_stopping: {patience: 5, metric: val_accuracy}

export:
  output_dir: exported_models
  formats:
    - tflite: {quantize: int8}
    - edgetpu: {}
```

```bash
# 3 comandos en lugar de horas de trabajo manual:
mlforge train --config road_signs_config.yaml
mlforge export --config road_signs_config.yaml --formats edgetpu
mlforge deploy edge --model exported_models/edgetpu/model_edgetpu.tflite \
                     --labels "stop,speed_40,speed_60,person,green_light"
```

El resultado: un modelo optimizado para Edge TPU + script de inferencia listo
para la Raspberry Pi, con la misma funcionalidad que el original pero:
- Entrenado con transfer learning (mejor accuracy)
- Data augmentation (más robusto)
- Quantizado automáticamente (mismo tamaño)
- Tracking de métricas (reproducible)
- Desplegable en cualquier plataforma (no solo Edge TPU)

## Resumen del Curso de Notebooks

| Notebook | Tema | Concepto clave |
|----------|------|---------|
| 01 | Redes Neuronales desde Cero | Forward prop, backprop, gradient descent |
| 02 | CNN Deep Dive | Convolución, pooling, arquitecturas |
| 03 | Transfer Learning | Feature extraction, fine-tuning |
| 04 | Optimización para Edge | Quantización, ONNX, TFLite |
| 05 | Deploy Everywhere | Android, iOS, Web, RPi+Coral |
| **06** | **DeepPiCar** | **Proyecto real: de concepto a producción** |

### Siguiente paso:
Ejecuta el ejemplo completo:
```bash
cd mlforge
mlforge train --config examples/01_image_classifier/config.yaml
mlforge export --config examples/01_image_classifier/config.yaml --formats all
mlforge deploy web --labels "tulipán,rosa,girasol,margarita,orquídea"
```